# CDF by tracer (DESI EDR)

This notebook reproduces the CDF calculation style from `r_values_plot.ipynb` and adds zone dispersion:
- Compute `r = (NDATA - NRAND) / (NDATA + NRAND)`
- Build one ECDF per zone on a shared `r` grid
- Plot the mean CDF across zones with zone-to-zone dispersion
- Plot separate curves for object (`ISDATA=True`) and random (`ISDATA=False`)

Using DESI files in `data/edr/raw` and `data/edr/classification` (same base data as `countfraction_redshift.ipynb`).


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
import pickle

plt.rcParams['font.family'] = 'Liberation Serif'
plt.rcParams['axes.linewidth'] = 1.2


In [2]:
def load_fits_df(path):
    tab = Table.read(path, format='fits', memmap=True)
    df = tab.to_pandas()

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: x.decode() if isinstance(x, (bytes, bytearray)) else x)
    return df


def get_zone_paths(raw_dir, class_dir, zone):
    raw_path = os.path.join(raw_dir, f'zone_{zone:02d}.fits.gz')
    cls_path = os.path.join(class_dir, f'zone_{zone:02d}_classified.fits.gz')
    return raw_path, cls_path


def compute_r_values(n_data, n_rand):
    n_data = np.asarray(n_data, dtype=float)
    n_rand = np.asarray(n_rand, dtype=float)
    denom = n_data + n_rand
    with np.errstate(divide='ignore', invalid='ignore'):
        r = np.where(denom != 0, (n_data - n_rand) / denom, np.nan)
    return r[np.isfinite(r)]


def tracer_mask(series, tracer):
    s = series.astype(str)
    if tracer == 'BGS_ANY':
        return s.str.startswith('BGS')
    return s.str.startswith(tracer)


def _ecdf_interp(arr, xgrid):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.full_like(xgrid, np.nan, dtype=float)
    a.sort()
    y = np.arange(1, a.size + 1) / a.size
    return np.interp(xgrid, a, y, left=0.0, right=1.0)


def build_cdf_stats_by_tracer(raw_dir, class_dir, zones, tracers=None, xbins=400):
    if tracers is None:
        tracers = ['BGS_ANY', 'LRG', 'ELG', 'QSO']

    xgrid = np.linspace(-1.0, 1.0, xbins)
    per_zone_data = {tr: [] for tr in tracers}
    per_zone_rand = {tr: [] for tr in tracers}

    for z in zones:
        _, cls_path = get_zone_paths(raw_dir, class_dir, z)
        cls_df = load_fits_df(cls_path)

        required = {'TRACERTYPE', 'ISDATA', 'NDATA', 'NRAND'}
        missing = required - set(cls_df.columns)
        if missing:
            raise KeyError(f'zone {z:02d} missing columns: {sorted(missing)}')

        for tr in tracers:
            m = tracer_mask(cls_df['TRACERTYPE'], tr)
            sub = cls_df[m]

            data_sub = sub[sub['ISDATA'] == True]
            rand_sub = sub[sub['ISDATA'] == False]

            if not data_sub.empty:
                r_data = compute_r_values(data_sub['NDATA'].to_numpy(), data_sub['NRAND'].to_numpy())
                if r_data.size:
                    y_data = _ecdf_interp(r_data, xgrid)
                    if np.isfinite(y_data).any():
                        per_zone_data[tr].append(y_data)

            if not rand_sub.empty:
                r_rand = compute_r_values(rand_sub['NDATA'].to_numpy(), rand_sub['NRAND'].to_numpy())
                if r_rand.size:
                    y_rand = _ecdf_interp(r_rand, xgrid)
                    if np.isfinite(y_rand).any():
                        per_zone_rand[tr].append(y_rand)

    stats_data = {}
    stats_rand = {}

    for tr in tracers:
        if per_zone_data[tr]:
            y_stack = np.vstack(per_zone_data[tr])
            mean = np.nanmean(y_stack, axis=0)
            var = np.nanvar(y_stack, axis=0)
            std = np.sqrt(np.maximum(var, 0.0))
        else:
            mean = np.array([])
            var = np.array([])
            std = np.array([])

        stats_data[tr] = {
            'TRACER': tr,
            'XGRID': xgrid,
            'MEAN': mean,
            'VAR': var,
            'STD': std,
            'N_ZONES': len(per_zone_data[tr]),
        }

        if per_zone_rand[tr]:
            y_stack = np.vstack(per_zone_rand[tr])
            mean = np.nanmean(y_stack, axis=0)
            var = np.nanvar(y_stack, axis=0)
            std = np.sqrt(np.maximum(var, 0.0))
        else:
            mean = np.array([])
            var = np.array([])
            std = np.array([])

        stats_rand[tr] = {
            'TRACER': tr,
            'XGRID': xgrid,
            'MEAN': mean,
            'VAR': var,
            'STD': std,
            'N_ZONES': len(per_zone_rand[tr]),
        }

    return stats_data, stats_rand


In [3]:
def plot_cdf_dispersion(stats_data, stats_rand, zones, out_dir, tracers=None, figsize=(14, 10)):
    if tracers is None:
        tracers = ['BGS_ANY', 'LRG', 'ELG', 'QSO']

    fig, ax = plt.subplots(figsize=figsize)
    ax.grid(linewidth=0.5, alpha=0.4, linestyle='--')

    tracer_color_map = {
        'BGS_ANY': 'blue',
        'LRG': 'green',
        'ELG': 'red',
        'QSO': 'purple',
    }

    tracer_name_map = {
        'BGS_ANY': 'BGS',
        'LRG': 'LRG',
        'ELG': 'ELG',
        'QSO': 'QSO',
    }

    for tracer in ['BGS_ANY', 'LRG', 'ELG', 'QSO']:
        color = tracer_color_map[tracer]
        tr_name = tracer_name_map[tracer]

        data_payload = stats_data.get(tracer, {})
        rand_payload = stats_rand.get(tracer, {})

        x_data = np.asarray(data_payload.get('XGRID', []), dtype=float)
        mean_data = np.asarray(data_payload.get('MEAN', []), dtype=float)
        std_data = np.asarray(data_payload.get('STD', []), dtype=float)
        if x_data.size and mean_data.size:
            ax.fill_between(x_data, mean_data - std_data, mean_data + std_data, alpha=0.15, color=color)
            ax.plot(x_data, mean_data, color=color, linewidth=2.5, label=f'{tr_name} object')

        x_rand = np.asarray(rand_payload.get('XGRID', []), dtype=float)
        mean_rand = np.asarray(rand_payload.get('MEAN', []), dtype=float)
        std_rand = np.asarray(rand_payload.get('STD', []), dtype=float)
        if x_rand.size and mean_rand.size:
            ax.fill_between(x_rand, mean_rand - std_rand, mean_rand + std_rand, alpha=0.10, color=color)
            ax.plot(x_rand, mean_rand, color=color, linewidth=1.2, linestyle='--', label=f'{tr_name} random')

    ax.axvline(x=0, color="black", linestyle=":", linewidth=1, alpha=0.5)
    ax.set_ylabel('CDF', fontsize=37)
    ax.set_xlabel(r"$r$", fontsize=37)
    ax.tick_params(axis="both", labelsize=32,pad = 10)
    fig.subplots_adjust(bottom=0.22)

    bb = ax.get_position()
    fig.legend(
        *ax.get_legend_handles_labels(),
        loc="upper left",
        ncol=4,
        frameon=True,
        fontsize=35,
        mode="expand",
        bbox_to_anchor=(bb.x0, bb.y0 - 0.12, bb.width, 0.001),
        bbox_transform=fig.transFigure,
        borderaxespad=0.0,
        columnspacing=5,
        handlelength=2.5,
    )

    os.makedirs(os.path.join(out_dir, "cdf"), exist_ok=True)
    path = '/Users/soguevaram/Desktop/Research/DESI/plots/cdf_dispersion_zones.png'
    plt.show()
    fig.savefig(path, dpi=360, bbox_inches='tight')
    print(f"Figura guardada en {path}")


In [8]:
base_dir = '/Users/soguevaram/Desktop/Research/DESI/data'
raw_dir = os.path.join(base_dir, 'edr/raw')
class_dir = os.path.join(base_dir, 'edr/classification')
out_dir = '/Users/soguevaram/Desktop/Research/DESI/data/results/cdf'

zones = range(20)
tracers = ['BGS_ANY', 'LRG', 'ELG', 'QSO']

stats_data, stats_rand = build_cdf_stats_by_tracer(
    raw_dir=raw_dir,
    class_dir=class_dir,
    zones=zones,
    tracers=tracers,
)

for tr in tracers:
    print(f"{tr}: data zones={stats_data[tr]['N_ZONES']} | random zones={stats_rand[tr]['N_ZONES']}")


BGS_ANY: data zones=20 | random zones=20
LRG: data zones=20 | random zones=20
ELG: data zones=20 | random zones=20
QSO: data zones=20 | random zones=20


In [9]:
# Save per-tracer mean CDF and dispersion to PKL files.
os.makedirs(out_dir, exist_ok=True)

for tr in tracers:
    tracer_data = stats_data[tr]
    tracer_rand = stats_rand[tr]

    with open(os.path.join(out_dir, f'{tr}_cdf_data.pkl'), 'wb') as f:
        pickle.dump(tracer_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(os.path.join(out_dir, f'{tr}_cdf_rand.pkl'), 'wb') as f:
        pickle.dump(tracer_rand, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'PKL files saved in: {out_dir}')


PKL files saved in: /Users/soguevaram/Desktop/Research/DESI/data/results/cdf
